# Polarization

```{autolink-concat}

```

The Dalitz-plot variables $(\sigma_1, \sigma_2, \sigma_3)$ describe the configuration of the three final-state momenta *within* the decay plane, but they say nothing about the spin states of the particles involved. In a Dalitz-plot-only fit, those spin states are not observed, so the helicities are summed **incoherently** in the intensity&mdash;an implicit integration over unmeasured degrees of freedom that averages away all interference between the different helicity sectors.

That information is not necessarily lost. If one of the final-state particles decays further, the directions of *its* decay products depend on its spin state, so the polarization of that particle becomes measurable. This notebook shows how to include such a subsequent decay in the amplitude model with {meth}`~ampform_dpd.DalitzPlotDecompositionBuilder.formulate`'s `polarized_final_states` argument, using $J/\psi \to K^0 \Sigma^+ \overline{p}$ with the weak decay $\Sigma^+ \to p \pi^0$ as an example. It turns out that this gives access to a model parameter that a Dalitz-plot-only fit cannot resolve at all.

In [ ]:
import logging
import os
import warnings

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import qrules
import sympy as sp
from IPython.display import Latex, Markdown
from tensorwaves.data.transform import SympyDataTransformer

from ampform_dpd import DalitzPlotDecompositionBuilder
from ampform_dpd.adapter.qrules import normalize_state_ids, to_three_body_decay
from ampform_dpd.dynamics.builder import formulate_breit_wigner_with_form_factor
from ampform_dpd.io import (
    as_markdown_table,
    aslatex,
    cached,
    mute_ampform_warnings,
    simplify_latex_rendering,
)
from ampform_dpd.polarization import (
    create_final_state_decay_angles,
    formulate_weak_decay_couplings,
)

simplify_latex_rendering()
logging.getLogger("absl").setLevel(logging.ERROR)  # mute JAX
warnings.simplefilter("ignore", category=RuntimeWarning)

if STATIC_PAGE := "EXECUTE_NB" in os.environ:
    mute_ampform_warnings()

## Decay definition

We take the same decay as in {doc}`jpsi2ksp`, but with only two interfering $\overline{\Sigma}{}^*$ resonances, in order to keep the expressions small. We will see [below](#effect-of-the-final-state-decay) why a single resonance would not suffice.

In [ ]:
REACTION = qrules.generate_transitions(
    initial_state=[("J/psi(1S)", [+1])],
    final_state=["K0", ("Sigma+", [+0.5]), ("p~", [+0.5])],
    allowed_interaction_types="strong",
    allowed_intermediate_particles=["Sigma(1660)", "Sigma(1670)"],
    formalism="canonical-helicity",
)
REACTION = normalize_state_ids(REACTION)
DECAY = to_three_body_decay(REACTION.transitions, min_ls=True)
Markdown(as_markdown_table([DECAY.initial_state, *DECAY.final_state.values()]))

In [ ]:
Latex(aslatex(DECAY, with_jp=True))

## Model formulation

The $\Sigma^+$ is final state $2$ in this decay. In a Dalitz-plot-only fit, its helicity $\lambda_2$ is summed incoherently, so amplitudes with different $\lambda_2$ never interfere. In this particular model, both resonances live in the same subsystem and we align to that subsystem, so there are no alignment Wigner rotations that could mix the $\lambda_2$ sectors either. Multiplying all production couplings that carry $\lambda_2=+\tfrac{1}{2}$ by a common phase $e^{i\delta}$ therefore leaves the Dalitz-plot intensity *exactly* invariant: $\delta$ is a flat direction of the fit. (In models with resonances in several subsystems, the alignment rotations do mix the sectors, and such a phase is only approximately flat.)

With the `polarized_final_states` argument, each amplitude gets one additional Wigner-$D$ function that couples $\lambda_2$ **coherently** to the helicity $\mu_2$ of the proton from $\Sigma^+ \to p\pi^0$,

$$
\tilde{A}_{\mu_2} = \sum_{\lambda_2} A_{\lambda_2}\, D^{1/2\,*}_{\lambda_2,\mu_2}\left(\phi_2, \theta_2, 0\right)\, \mathcal{H}^\mathrm{fs}_{\mu_2}\,,
$$

where $(\phi_2, \theta_2)$ is the direction of the proton in the aligned rest frame of the $\Sigma^+$. It is now $\mu_2$ that is summed incoherently in the intensity (see {meth}`~ampform_dpd.DalitzPlotDecompositionBuilder.formulate_final_state_decay`):

In [ ]:
model_builder = DalitzPlotDecompositionBuilder(DECAY, min_ls=True)
for chain in model_builder.decay.chains:
    model_builder.dynamics_choices.register_builder(
        chain, formulate_breit_wigner_with_form_factor
    )
model = model_builder.formulate(reference_subsystem=2, polarized_final_states=[2])
model.intensity

The angles $(\phi_2, \theta_2)$ are new kinematic variables of the model. They cannot be computed from the Mandelstam variables, because the proton momentum lies outside the three-body decay plane. The model therefore defines them in terms of four-momenta&mdash;`p1`, `p2`, `p3` in the center-of-momentum frame and `q2` for the proton&mdash;through a chain of boosts and rotations with {func}`ampform.kinematics.angles.compute_helicity_angles` (see {func}`~ampform_dpd.polarization.formulate_final_state_decay_angles`):

In [ ]:
decay_angles = create_final_state_decay_angles(state_id=2)
Latex(aslatex({symbol: model.variables[symbol] for symbol in decay_angles}))

The final-state decay couplings $\mathcal{H}^\mathrm{fs}$ default to $1$, in which case the intensity is insensitive to the $\Sigma^+$ polarization. For a parity-violating two-body decay of a spin-½ particle, they are fixed by the decay-asymmetry parameter $\alpha$. The $\Sigma^+ \to p\pi^0$ decay has $\alpha = -0.98$: in the $\Sigma^+$ rest frame, the proton is emitted with probability $\propto 1 + \alpha\, \vec{P}_\Sigma \cdot \hat{n}_p$, that is, almost fully anti-parallel to the $\Sigma^+$ spin. The proton direction is therefore an almost perfect measurement of the $\Sigma^+$ spin state.

In [ ]:
Σ = DECAY.final_state[2]
α = sp.Symbol("alpha", real=True)
weak_couplings = formulate_weak_decay_couplings(Σ, α)
Latex(aslatex(weak_couplings))

## Preparing for input data

Instead of sampling the angles directly, we generate four-momenta, as would be available in a real analysis: a flat distribution over the Dalitz plane with isotropic orientation, followed by an isotropic $\Sigma^+ \to p\pi^0$ decay. The kinematic variables&mdash;including $(\phi_2, \theta_2)$ through the boost-chain expressions above&mdash;are then computed from these momenta.

In [ ]:
def generate_phase_space(n_events: int, seed: int) -> dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    m0, m1, m2, m3 = (float(v) for v in model.masses.values())
    σ1 = rng.uniform((m2 + m3) ** 2, (m0 - m1) ** 2, n_events)
    σ2 = rng.uniform((m1 + m3) ** 2, (m0 - m2) ** 2, n_events)
    σ3 = m0**2 + m1**2 + m2**2 + m3**2 - σ1 - σ2
    energies = [
        (m0**2 + m**2 - σ) / (2 * m0) for m, σ in [(m1, σ1), (m2, σ2), (m3, σ3)]
    ]
    with np.errstate(invalid="ignore"):
        p_norms = [
            np.sqrt(E**2 - m**2) for E, m in zip(energies, (m1, m2, m3), strict=True)
        ]
        cos_12 = (2 * energies[0] * energies[1] + m1**2 + m2**2 - σ3) / (
            2 * p_norms[0] * p_norms[1]
        )
        selector = np.all(
            [m < E for E, m in zip(energies, (m1, m2, m3), strict=True)], axis=0
        )
        selector &= np.abs(cos_12) < 1
        σ1, σ2, σ3 = σ1[selector], σ2[selector], σ3[selector]
        energies = [E[selector] for E in energies]
        p_norms = [p[selector] for p in p_norms]
        cos_12 = cos_12[selector]
    n = selector.sum()
    zeros = np.zeros(n)
    sin_12 = np.sqrt(1 - cos_12**2)
    vec1 = np.array([zeros, zeros, p_norms[0]]).T
    vec2 = (p_norms[1] * np.array([sin_12, zeros, cos_12])).T
    vec3 = -vec1 - vec2
    rotation = _random_rotation(n, rng)
    vec1, vec2, vec3 = (
        np.einsum("nij,nj->ni", rotation, v) for v in (vec1, vec2, vec3)
    )
    p2 = np.column_stack([energies[1], vec2])
    q2 = _two_body_decay_products(
        p2, m2, m_daughter=0.93827, m_other=0.1349768, rng=rng
    )
    return {
        "sigma1": σ1,
        "sigma2": σ2,
        "sigma3": σ3,
        "p1": np.column_stack([energies[0], vec1]),
        "p2": p2,
        "p3": np.column_stack([energies[2], vec3]),
        "q2": q2,
    }


def _random_rotation(n: int, rng: np.random.Generator) -> np.ndarray:
    α = rng.uniform(-np.pi, +np.pi, n)
    β = np.arccos(rng.uniform(-1, +1, n))
    γ = rng.uniform(-np.pi, +np.pi, n)
    return np.einsum("nij,njk,nkl->nil", _rotation_z(α), _rotation_y(β), _rotation_z(γ))


def _rotation_y(angle: np.ndarray) -> np.ndarray:
    n = len(angle)
    matrix = np.zeros((n, 3, 3))
    matrix[:, 0, 0] = matrix[:, 2, 2] = np.cos(angle)
    matrix[:, 0, 2] = np.sin(angle)
    matrix[:, 2, 0] = -np.sin(angle)
    matrix[:, 1, 1] = 1
    return matrix


def _rotation_z(angle: np.ndarray) -> np.ndarray:
    n = len(angle)
    matrix = np.zeros((n, 3, 3))
    matrix[:, 0, 0] = matrix[:, 1, 1] = np.cos(angle)
    matrix[:, 0, 1] = -np.sin(angle)
    matrix[:, 1, 0] = np.sin(angle)
    matrix[:, 2, 2] = 1
    return matrix


def _two_body_decay_products(
    parent: np.ndarray,
    parent_mass: float,
    m_daughter: float,
    m_other: float,
    rng: np.random.Generator,
) -> np.ndarray:
    n = len(parent)
    q_norm = np.sqrt(
        (parent_mass**2 - (m_daughter + m_other) ** 2)
        * (parent_mass**2 - (m_daughter - m_other) ** 2)
    ) / (2 * parent_mass)
    cos_θ = rng.uniform(-1, +1, n)
    sin_θ = np.sqrt(1 - cos_θ**2)
    φ = rng.uniform(-np.pi, +np.pi, n)
    q_star = q_norm * np.array([sin_θ * np.cos(φ), sin_θ * np.sin(φ), cos_θ]).T
    E_star = np.full(n, np.sqrt(m_daughter**2 + q_norm**2))
    direction = parent[:, 1:] / np.linalg.norm(parent[:, 1:], axis=1)[:, None]
    γ = parent[:, 0] / parent_mass
    βγ = np.linalg.norm(parent[:, 1:], axis=1) / parent_mass
    q_parallel = np.einsum("ni,ni->n", q_star, direction)
    energy = γ * E_star + βγ * q_parallel
    momentum = q_star + ((γ - 1) * q_parallel + βγ * E_star)[:, None] * direction
    return np.column_stack([energy, momentum])


phsp = generate_phase_space(n_events=500_000, seed=1)
transformer = SympyDataTransformer.from_sympy(
    {symbol: expr.xreplace(model.masses) for symbol, expr in model.variables.items()},
    backend="jax",
)
phsp.update(transformer(phsp))
{key: array.shape for key, array in phsp.items()}

With the default couplings $\mathcal{H}^\mathrm{fs} = 1$, the intensity coincides with the Dalitz-plot-only model for *any* proton direction (unitarity of the Wigner-$D$ functions):

In [ ]:
intensity_expr = cached.unfold(model)
default_func = cached.lambdify(
    cached.xreplace(intensity_expr, model.parameter_defaults),
    backend="jax",
)
dalitz_only_model = model_builder.formulate(reference_subsystem=2)
dalitz_only_func = cached.lambdify(
    cached.xreplace(
        cached.unfold(dalitz_only_model), dalitz_only_model.parameter_defaults
    ),
    backend="jax",
)
np.testing.assert_allclose(
    np.real(default_func(phsp)),
    np.real(dalitz_only_func(phsp)),
)

More generally, integrating the intensity over the proton direction always returns the Dalitz-plot-only intensity, for *any* choice of couplings (orthogonality of the Wigner-$D$ functions). Omitting the final-state decay from the model is therefore equivalent to **integrating this degree of freedom away**: the information that the proton direction carries about the $\Sigma^+$ spin is averaged out.

To see that this information makes a difference, we insert the weak-decay couplings and rotate the $\lambda_2=+\tfrac{1}{2}$ production couplings of both resonances by a phase $e^{i\delta}$:

In [ ]:
def create_intensity_func(delta: float, alpha_value: float = -0.98):
    parameter_values = dict(model.parameter_defaults)
    for symbol in parameter_values:
        is_production_coupling = isinstance(symbol, sp.Indexed) and "production" in str(
            symbol.base
        )
        if is_production_coupling and symbol.indices[2] == sp.Rational(1, 2):
            parameter_values[symbol] *= np.exp(1j * delta)
    coupling_values = {
        symbol: complex(expr.xreplace({α: alpha_value}))
        for symbol, expr in weak_couplings.items()
    }
    substituted_expr = cached.xreplace(intensity_expr, coupling_values)
    return cached.lambdify(
        cached.xreplace(substituted_expr, parameter_values), backend="jax"
    )


intensities = {
    label: jnp.real(create_intensity_func(delta)(phsp))
    for label, delta in {
        R"$\delta = 0$": 0,
        R"$\delta = \pi/2$": np.pi / 2,
        R"$\delta = \pi$": np.pi,
    }.items()
}

## Effect of the final-state decay

The three models produce *identical* distributions over the Dalitz plane&hellip;

In [ ]:
%config InlineBackend.figure_formats = ['svg']
plt.rc("font", size=12)
fig, axes = plt.subplots(figsize=(12, 4), ncols=2, layout="constrained")
for ax, (sigma_key, x_label) in zip(
    axes,
    {
        "sigma2": R"$\sigma_2 = M^2\left(K^0 \overline{p}\right)$",
        "sigma3": R"$\sigma_3 = M^2\left(K^0 \Sigma^+\right)$",
    }.items(),
    strict=True,
):
    for label, weights in intensities.items():
        bin_values, bin_edges = np.histogram(
            phsp[sigma_key], bins=80, weights=weights, density=True
        )
        ax.stairs(bin_values, bin_edges, label=label, lw=2)
    ax.set_xlabel(x_label)
    ax.set_ylim(0, None)
axes[0].set_ylabel("Normalized intensity (a.u.)")
axes[-1].legend(fontsize=10)
plt.show()

&hellip;but the weak decay converts the invisible phase $\delta$ into a visible modulation of the proton azimuthal angle $\phi_2$ in the $\Sigma^+$ rest frame. The modulation stems from the interference between the $\lambda_2 = \pm\tfrac{1}{2}$ amplitudes, which corresponds to a *transverse* polarization of the $\Sigma^+$. A *longitudinal* polarization would instead require unequal coupling magnitudes between the two $\lambda_2$ sectors and would tilt the $\cos\theta_2$ distribution; since we only rotate phases here, that distribution stays flat. The interference term is also the reason why two resonances are needed: within a single chain, it is weighted by the difference of the squared $\overline{\Sigma}{}^* \to \overline{p}K^0$ decay couplings, summed over the unobserved $\overline{p}$ helicity, and parity conservation of that strong decay forces this difference to vanish. Interference between two *different* resonances&mdash;here even of opposite parity&mdash;escapes this cancellation. A fit that includes the proton direction can therefore determine $\delta$, at no cost in extra parameters, since the $\mathcal{H}^\mathrm{fs}$ couplings are fixed by $\alpha$:

In [ ]:
angular_variables = {
    "theta_2": (np.asarray(np.cos(phsp["theta_2"])), R"$\cos\theta_2$"),
    "phi_2": (np.asarray(phsp["phi_2"]), R"$\phi_2$"),
}
fig, axes = plt.subplots(figsize=(12, 4), ncols=2, layout="constrained")
for ax, (x, x_label) in zip(axes, angular_variables.values(), strict=True):
    for label, weights in intensities.items():
        bin_values, bin_edges = np.histogram(x, bins=50, weights=weights, density=True)
        ax.stairs(bin_values, bin_edges, label=label, lw=2)
    ax.set_xlabel(x_label)
    ax.set_ylim(0, None)
axes[0].set_ylabel("Normalized intensity (a.u.)")
axes[-1].legend(fontsize=10)
plt.show()

:::{seealso}
{meth}`~ampform_dpd.DalitzPlotDecompositionBuilder.formulate_final_state_decay` and the {mod}`ampform_dpd.polarization` module for more information about the implementation.
:::